<a href="https://colab.research.google.com/github/chanvaram/korean_handwriting_font/blob/main/colab_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖊️ 한글 손글씨 폰트 생성기
**4글자 손글씨 → 11,172자 전체 한글 폰트 자동 생성**

순서대로 셀을 실행하세요. (Shift+Enter)

---
### ⚠️ 시작 전 확인
상단 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU** 선택

In [1]:
# ✅ Cell 1 — GPU 확인
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 모델:', torch.cuda.get_device_name(0))
else:
    print('⚠️ GPU가 없습니다. 런타임 유형을 T4 GPU로 변경하세요.')

GPU 사용 가능: True
GPU 모델: Tesla T4


In [2]:
# ✅ Cell 2 — 레포 클론 & LF-Font 설치
import os

# 내 레포 클론
if not os.path.exists('korean_handwriting_font'):
    !git clone https://github.com/chanvaram/korean_handwriting_font.git
%cd korean_handwriting_font

# LF-Font 클론 (extern/)
if not os.path.exists('extern/lffont'):
    !git clone https://github.com/clovaai/lffont.git extern/lffont

print('✅ 클론 완료')

Cloning into 'korean_handwriting_font'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 27 (delta 4), reused 24 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 18.10 KiB | 4.52 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/korean_handwriting_font
Cloning into 'extern/lffont'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 224 (delta 53), reused 37 (delta 37), pack-reused 150 (from 1)
Receiving objects: 100% (224/224), 56.59 MiB | 30.88 MiB/s, done.
Resolving deltas: 100% (115/115), done.
✅ 클론 완료


In [3]:
# ✅ Cell 3 — 의존성 설치
!pip install -q sconf lmdb msgpack ruamel.yaml jsonlib-python3
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q Pillow numpy opencv-python tqdm fonttools

# potrace, fontforge 설치
!apt-get install -qq potrace fontforge python3-fontforge

print('✅ 설치 완료')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.6 MB/s eta 0:00:00
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../00-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../01-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../02-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../03-gsettings-desktop-schemas_42.0-1ubuntu1_all.de

In [ ]:
# ✅ Cell 4 — 사전학습 가중치 다운로드
# LF-Font GitHub 에서 pretrained 모델 링크 확인 후 아래 URL 교체
# https://github.com/clovaai/lffont → Releases 또는 README 참고

import os
os.makedirs('pretrained', exist_ok=True)

# 가중치 파일이 Google Drive에 있는 경우
# !gdown --id <FILE_ID> -O pretrained/lffont.pth

# 직접 URL이 있는 경우
# !wget -q <URL> -O pretrained/lffont.pth

# 확인
if os.path.exists('pretrained/lffont.pth'):
    size = os.path.getsize('pretrained/lffont.pth') / 1e6
    print(f'✅ 가중치 로드 완료: {size:.1f} MB')
else:
    print('⚠️ pretrained/lffont.pth 없음 — Cell 4 주석 확인 후 다운로드')

In [ ]:
# ✅ Cell 5 — 손글씨 샘플 업로드
# 아래 4글자를 손으로 써서 스캔/촬영 후 업로드하세요
# 파일명: 갈.png  넓.png  읽.png  좋.png

from google.colab import files
import os

os.makedirs('data/my_handwriting', exist_ok=True)

print('업로드할 파일: 갈.png, 넓.png, 읽.png, 좋.png')
print('파일 선택 창이 열립니다...')

uploaded = files.upload()

for fname, data in uploaded.items():
    dest = f'data/my_handwriting/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  저장: {dest}')

print(f'\n✅ {len(uploaded)}장 업로드 완료')

In [ ]:
# ✅ Cell 6 — 이미지 전처리 & 미리보기
import sys
sys.path.insert(0, '.')

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

def preprocess(img_path, size=128):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = cv2.findNonZero(binary)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        pad = 15
        x, y = max(0,x-pad), max(0,y-pad)
        w = min(binary.shape[1]-x, w+2*pad)
        h = min(binary.shape[0]-y, h+2*pad)
        binary = binary[y:y+h, x:x+w]
    sz = max(binary.shape)
    padded = np.zeros((sz,sz), dtype=np.uint8)
    ph, pw = (sz-binary.shape[0])//2, (sz-binary.shape[1])//2
    padded[ph:ph+binary.shape[0], pw:pw+binary.shape[1]] = binary
    resized = cv2.resize(padded, (size,size), interpolation=cv2.INTER_AREA)
    return cv2.bitwise_not(resized)

os.makedirs('data/preprocessed', exist_ok=True)
src_files = [f for f in os.listdir('data/my_handwriting') if f.endswith('.png')]

fig, axes = plt.subplots(2, len(src_files), figsize=(3*len(src_files), 6))
if len(src_files) == 1:
    axes = [[axes[0]], [axes[1]]]

for i, fname in enumerate(src_files):
    src = f'data/my_handwriting/{fname}'
    dst = f'data/preprocessed/{fname}'
    result = preprocess(src)
    cv2.imwrite(dst, result)

    orig = cv2.imread(src, cv2.IMREAD_GRAYSCALE)
    axes[0][i].imshow(orig, cmap='gray')
    axes[0][i].set_title(f'원본: {fname}', fontsize=10)
    axes[0][i].axis('off')

    axes[1][i].imshow(result, cmap='gray')
    axes[1][i].set_title('전처리 후', fontsize=10)
    axes[1][i].axis('off')

plt.tight_layout()
plt.show()
print(f'✅ {len(src_files)}장 전처리 완료 → data/preprocessed/')

In [ ]:
# ✅ Cell 7 — 폰트 생성 (LF-Font inference)
import sys
sys.path.insert(0, 'extern/lffont')

# LF-Font 공식 inference 실행
# (lffont 레포의 실제 inference 스크립트 사용)
!python extern/lffont/inference.py \
    --config extern/lffont/cfgs/lffont.yaml \
    --weight pretrained/lffont.pth \
    --ref_path data/preprocessed \
    --output_path output/png

import os
generated = len([f for f in os.listdir('output/png') if f.endswith('.png')])
print(f'\n✅ 생성 완료: {generated}자 / 11,172자')

In [ ]:
# ✅ Cell 8 — 생성 결과 미리보기 (랜덤 샘플 36자)
import os, random
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image

png_dir = 'output/png'
all_pngs = [f for f in os.listdir(png_dir) if f.endswith('.png')]
samples = random.sample(all_pngs, min(36, len(all_pngs)))

fig, axes = plt.subplots(6, 6, figsize=(12, 12))
for ax, fname in zip(axes.flat, samples):
    img = Image.open(os.path.join(png_dir, fname)).convert('L')
    ax.imshow(img, cmap='gray')
    ax.set_title(fname.replace('.png',''), fontsize=14)
    ax.axis('off')

plt.suptitle('생성된 손글씨 샘플 (36자)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('output/preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 미리보기 저장: output/preview.png')

In [ ]:
# ✅ Cell 9 — PNG → TTF 변환
import subprocess, os

os.makedirs('output/svg', exist_ok=True)

# PNG → BMP → SVG (potrace)
png_files = [f for f in os.listdir('output/png') if f.endswith('.png')]
print(f'벡터화 중... ({len(png_files)}자)')

from PIL import Image
import numpy as np
from tqdm.notebook import tqdm

errors = []
for fname in tqdm(png_files):
    char = fname.replace('.png', '')
    png_path = f'output/png/{fname}'
    bmp_path = f'/tmp/{char}.bmp'
    svg_path = f'output/svg/{char}.svg'

    img = Image.open(png_path).convert('L')
    arr = np.array(img)
    binary = (arr < 128).astype(np.uint8) * 255
    Image.fromarray(binary).convert('1').save(bmp_path)

    result = subprocess.run(
        ['potrace', bmp_path, '-s', '-o', svg_path, '--tight'],
        capture_output=True
    )
    if result.returncode != 0:
        errors.append(char)

print(f'\nSVG 변환 완료: {len(png_files)-len(errors)}/{len(png_files)}자')

# SVG → TTF (fontforge)
print('TTF 빌드 중...')
!python scripts/build_ttf.py \
    --png_dir output/png \
    --svg_dir output/svg \
    --output output/my_handwriting.ttf \
    --font_name MyHandwriting

if os.path.exists('output/my_handwriting.ttf'):
    size = os.path.getsize('output/my_handwriting.ttf') / 1e6
    print(f'\n✅ TTF 생성 완료: output/my_handwriting.ttf ({size:.1f} MB)')
else:
    print('⚠️ TTF 생성 실패 — Cell 9 오류 메시지 확인')

In [ ]:
# ✅ Cell 10 — 다운로드
from google.colab import files
import os

# TTF 폰트 다운로드
if os.path.exists('output/my_handwriting.ttf'):
    files.download('output/my_handwriting.ttf')
    print('✅ my_handwriting.ttf 다운로드 시작')

# 미리보기 이미지 다운로드
if os.path.exists('output/preview.png'):
    files.download('output/preview.png')
    print('✅ preview.png 다운로드 시작')